# Post-Training Quantization Analysis â€” ResNet-20 on CIFAR-10

Compares accuracy and inference latency across INT16, INT8, and INT4 weight quantization.


In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
import sys
sys.path.append('Quantization_project')

import torch
import torch.nn as nn
import copy, time
import numpy as np
import matplotlib.pyplot as plt
from resnet import resnet20, BasicBlock

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

In [ ]:
MODEL_PATH = 'Quantization_project/resnet20_base_best.pth'

net_fp32 = resnet20()
net_fp32.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
net_fp32.eval()
print('Loaded FP32 model.')

In [ ]:
def fold_bn(conv, bn):
    w = conv.weight.data.clone()
    b = torch.zeros(w.size(0))
    gamma, beta = bn.weight.data, bn.bias.data
    mean, var, eps = bn.running_mean, bn.running_var, bn.eps
    std = torch.sqrt(var + eps)
    conv.weight.data = w * (gamma / std).view(-1, 1, 1, 1)
    conv.bias = nn.Parameter(gamma * (b - mean) / std + beta)
    bn.weight.data.fill_(1.0)
    bn.bias.data.fill_(0.0)
    bn.running_mean.fill_(0.0)
    bn.running_var.fill_(1.0)

def fold_model_bn(model):
    model = copy.deepcopy(model)
    model.eval()
    fold_bn(model.conv1, model.bn1)
    for layer in [model.layer1, model.layer2, model.layer3]:
        for block in layer:
            fold_bn(block.conv1, block.bn1)
            fold_bn(block.conv2, block.bn2)
            if len(block.shortcut) > 0:
                fold_bn(block.shortcut[0], block.shortcut[1])
    return model

def quantize_tensor(t, num_bits):
    qmin, qmax = -(2**(num_bits-1)), 2**(num_bits-1) - 1
    min_val, max_val = t.min(), t.max()
    scale = (max_val - min_val) / (qmax - qmin)
    scale = max(scale.item(), 1e-8)
    zero_point = qmin - round(min_val.item() / scale)
    t_q = torch.clamp(torch.round(t / scale + zero_point), qmin, qmax)
    return (t_q - zero_point) * scale

def quantize_model(model, num_bits):
    model_q = fold_model_bn(model)
    for name, param in model_q.named_parameters():
        if 'weight' in name and param.dim() > 1:
            param.data = quantize_tensor(param.data, num_bits)
    return model_q

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100. * correct / total

def measure_latency(model, loader, runs=30):
    model.eval()
    with torch.no_grad():
        for inputs, _ in loader:
            _ = model(inputs)

    times = []
    with torch.no_grad():
        for _ in range(runs):
            start = time.time()
            for inputs, _ in loader:
                _ = model(inputs)
            times.append(time.time() - start)
    return np.mean(times), np.std(times)

In [ ]:
bit_widths = {'FP32': None, 'INT16': 16, 'INT8': 8, 'INT4': 4}
results = {}

for name, bits in bit_widths.items():
    print(f'\nEvaluating {name}...')
    if bits is None:
        model = fold_model_bn(net_fp32)
    else:
        model = quantize_model(net_fp32, bits)
    acc = evaluate(model, testloader)
    avg_time, std_time = measure_latency(model, testloader, runs=30)
    results[name] = {'accuracy': acc, 'avg_time': avg_time, 'std_time': std_time, 'bits': bits or 32}
    print(f'  Accuracy: {acc:.2f}%  |  Latency: {avg_time:.4f} Â± {std_time:.4f}s')

total_params = sum(p.numel() for p in net_fp32.parameters())
print(f'\nTotal parameters: {total_params:,}')
print(f'\n{"Config":<8} {"Accuracy":>10} {"Latency (s)":>14} {"Model Size (MB)":>16}')
print('-' * 52)
for name, r in results.items():
    size_mb = total_params * r["bits"] / 8 / 1e6
    print(f'{name:<8} {r["accuracy"]:>9.2f}% {r["avg_time"]:>10.4f}Â±{r["std_time"]:.4f} {size_mb:>14.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
times = [results[n]['avg_time'] for n in names]
stds = [results[n]['std_time'] for n in names]
bits = [results[n]['bits'] for n in names]

axes[0].bar(names, accs, color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Test Accuracy vs Quantization Level')
axes[0].set_ylim(min(accs) - 5, max(accs) + 2)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

axes[1].bar(names, times, yerr=stds, color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'], capsize=4)
axes[1].set_ylabel('Inference Time (s)')
axes[1].set_title('Inference Latency vs Quantization Level')

plt.tight_layout()
plt.savefig('quantization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
bit_widths = list(range(16, 1, -1))
accs = []

for bits in bit_widths:
    model_q = quantize_model(net_fp32, bits)
    acc = evaluate(model_q, testloader)
    accs.append(acc)
    print(f'INT{bits}: {acc:.2f}%')

fp32_acc = evaluate(fold_model_bn(net_fp32), testloader)
print(f'\nFP32 baseline: {fp32_acc:.2f}%')

plt.figure(figsize=(10, 5))
plt.plot(bit_widths, accs, 'o-', color='#2196F3', linewidth=2, markersize=8)
plt.axhline(y=fp32_acc, color='#F44336', linestyle='--', label=f'FP32 ({fp32_acc:.1f}%)')
plt.xlabel('Bit Width')
plt.ylabel('Test Accuracy (%)')
plt.title('Accuracy vs Quantization Bit Width')
plt.xticks(bit_widths)
plt.gca().invert_xaxis()
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_vs_quantization.png', dpi=150, bbox_inches='tight')
plt.show()